## 1. 원본 로딩 및 강남구 추출

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 운영체제에 설치된 한글 글꼴을 우선순위에 따라 선택합니다.
korean_fonts = ("Malgun Gothic", "Apple SD Gothic Neo", "NanumGothic",
                "Noto Sans CJK KR", "Noto Sans KR")
installed_fonts = set(font_manager.get_font_names())
korean_font = next((name for name in korean_fonts if name in installed_fonts), None)
if korean_font is None:
    raise RuntimeError("한글 그래프를 그리려면 맑은 고딕, 나눔고딕 또는 Noto Sans CJK KR을 설치하세요.")
plt.rcParams.update({"font.family": korean_font, "axes.unicode_minus": False})
from IPython.display import display

DATASET_KEY = '국민건강보험공단_연도별 시군구별 천식 관련 진료인원_20250101'

def find_repo():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if path.name == 'do_not_lung' and (path/'notebooks').is_dir():
            return path
        if (path/'do_not_lung'/'notebooks').is_dir():
            return path/'do_not_lung'
    raise FileNotFoundError('do_not_lung 저장소를 찾지 못했습니다.')

REPO = find_repo()
DATA_ROOT = REPO.parent/'data'
RAW_ROOT = DATA_ROOT/'raw_data'
OUT_DIR = DATA_ROOT/'processed_data'/DATASET_KEY

# 사용자의 로컬 raw_data 우선. 이 대화의 업로드 파일처럼 상위 폴더에 있는 경우에도 탐색.
pattern = re.compile(r'^'+re.escape(DATASET_KEY)+r'(?:\(\d+\))?\.csv$')
possible = []
if RAW_ROOT.exists():
    possible += [p for p in RAW_ROOT.rglob('*.csv') if pattern.fullmatch(p.name)]
possible += [p for p in REPO.parent.glob('*.csv') if pattern.fullmatch(p.name)]
possible = sorted(set(possible), key=lambda p: (p.name != DATASET_KEY+'.csv', str(p)))
if not possible:
    raise FileNotFoundError('원본 CSV를 data/raw_data 또는 작업 상위폴더에서 찾지 못했습니다.')
RAW_PATH = possible[0]

for encoding in ('utf-8-sig', 'cp949', 'euc-kr'):
    try:
        raw = pd.read_csv(RAW_PATH, encoding=encoding, dtype='string')
        break
    except UnicodeDecodeError:
        continue
else:
    raise UnicodeError('원본 CSV 인코딩 판독 실패')

raw.columns = raw.columns.str.strip()
raw_cols = raw.columns.tolist()
for c in raw_cols:
    raw[c] = raw[c].str.strip().replace('', pd.NA)

required = ['진료개시년도', '시도', '시도시군구', '데이터기준일']
missing_required = sorted(set(required)-set(raw_cols))
if missing_required:
    raise ValueError(f'필수 컬럼 없음: {missing_required}')

AGE_COLS = [c for c in raw_cols if re.fullmatch(r'\d+-\d+세|\d+세이상', c)]
if not AGE_COLS:
    raise ValueError('연령대별 진료인원 컬럼이 없습니다.')

mask = raw['시도'].eq('서울특별시') & raw['시도시군구'].eq('강남구')
selected = raw.loc[mask].copy()
selected.insert(0, 'source_row_id', selected.index.astype('int64')+1)
selected = selected.reset_index(drop=True)
if selected.empty:
    raise ValueError('강남구(서울특별시) 행이 존재하지 않습니다.')

print('원본 파일:', RAW_PATH.name, '| 인코딩:', encoding)
print('원본 구조:', raw.shape, '| 강남구:', selected.shape)
print('강남구 원본 컬럼:', len(raw_cols), '| 연령대 컬럼:', len(AGE_COLS))
print('메모리 사용량 (MB):', round(selected.memory_usage(deep=True).sum()/1024**2, 4))
print('연도별 전국 원본 행 수:', raw['진료개시년도'].value_counts().sort_index().to_dict())
print('강남구 연도:', selected['진료개시년도'].tolist())
print('강남구 기준일:', selected['데이터기준일'].unique().tolist())
display(selected.head())
display(selected.dtypes.rename('dtype').to_frame())

원본 파일: 국민건강보험공단_연도별 시군구별 천식 관련 진료인원_20250101.csv | 인코딩: cp949
원본 구조: (500, 22) | 강남구: (2, 23)
강남구 원본 컬럼: 22 | 연령대 컬럼: 18
메모리 사용량 (MB): 0.0024
연도별 전국 원본 행 수: {'2020': 250, '2021': 250}
강남구 연도: ['2020', '2021']
강남구 기준일: ['2025-01-01']


,source_row_id,진료개시년도,시도,시도시군구,0-4세,5-9세,10-14세,15-19세,20-24세,25-29세,...,45-49세,50-54세,55-59세,60-64세,65-69세,70-74세,75-79세,80-84세,85세이상,데이터기준일
0,23,2020,서울특별시,강남구,660,799,366,234,251,349,...,505,444,367,494,461,378,325,192,169,2025-01-01
1,273,2021,서울특별시,강남구,778,729,370,279,199,252,...,391,406,320,354,404,322,285,173,140,2025-01-01


,dtype
source_row_id,int64
진료개시년도,string
시도,string
시도시군구,string
0-4세,string
5-9세,string
10-14세,string
15-19세,string
20-24세,string
25-29세,string


## 2. 결측·중복·형식 및 비식별값 검증

In [2]:
clean = selected.copy()
raw_age_values = selected[AGE_COLS].copy()
suppressed = raw_age_values.eq('*').fillna(False)
invalid_values = {}
negative_values = {}
non_integer_values = {}

for col in AGE_COLS:
    parsed = pd.to_numeric(raw_age_values[col].mask(suppressed[col]), errors='coerce')
    invalid_values[col] = (raw_age_values[col].notna() & ~suppressed[col] & parsed.isna()).fillna(False)
    negative_values[col] = parsed.lt(0).fillna(False)
    non_integer_values[col] = (parsed.notna() & parsed.mod(1).ne(0)).fillna(False)
    # 결측 또는 비식별(*)은 NA로 유지하되 원본 비식별 여부를 별도로 보관한다.
    clean[col] = parsed.astype('Float64')
    clean[col+'_suppressed_flag'] = suppressed[col].astype(bool)

clean['진료개시년도'] = pd.to_numeric(clean['진료개시년도'], errors='coerce').astype('Int64')
parsed_baseline = pd.to_datetime(selected['데이터기준일'], errors='coerce')
clean['데이터기준일'] = parsed_baseline.dt.strftime('%Y-%m-%d').astype('string')
clean['valid_year_flag'] = (
    clean['진료개시년도'].ge(1900) &
    clean['진료개시년도'].le(parsed_baseline.dt.year)
).fillna(False).astype(bool)
clean['valid_reference_date_flag'] = parsed_baseline.notna().astype(bool)
key_cols = ['진료개시년도', '시도', '시도시군구']
clean['duplicate_record_flag'] = selected.duplicated(subset=raw_cols, keep=False).astype(bool)
clean['duplicate_key_flag'] = clean.duplicated(subset=key_cols, keep=False).astype(bool)
clean['suppressed_age_band_count'] = suppressed.sum(axis=1).astype('int64')
clean['missing_age_band_count'] = clean[AGE_COLS].isna().sum(axis=1).astype('int64')
clean['invalid_age_count_flag'] = pd.DataFrame(invalid_values).any(axis=1).astype(bool)
clean['negative_age_count_flag'] = pd.DataFrame(negative_values).any(axis=1).astype(bool)
clean['non_integer_age_count_flag'] = pd.DataFrame(non_integer_values).any(axis=1).astype(bool)
clean['review_needed'] = (
    ~clean['valid_year_flag'] | ~clean['valid_reference_date_flag'] |
    clean['duplicate_record_flag'] | clean['duplicate_key_flag'] |
    clean['invalid_age_count_flag'] | clean['negative_age_count_flag'] |
    clean['non_integer_age_count_flag']
).astype(bool)

missing_source = selected[raw_cols].isna().sum()
missing_after = clean[AGE_COLS].isna().sum()
print('강남구 정확히 동일한 행:', int(clean['duplicate_record_flag'].sum()))
print('연도·시도·시군구 복합키 중복:', int(clean['duplicate_key_flag'].sum()))
print('강남구 비식별(*) 셀:', int(clean['suppressed_age_band_count'].sum()))
print('강남구 원본 공란 셀:', int(selected[raw_cols].isna().sum().sum()))
print('잘못된/음수/소수 인원값이 있는 행:',
      int(clean['invalid_age_count_flag'].sum()),
      int(clean['negative_age_count_flag'].sum()),
      int(clean['non_integer_age_count_flag'].sum()))
print('검토 필요 행:', int(clean['review_needed'].sum()))
print('전국 원본 비식별(*) 셀:', int(raw[AGE_COLS].eq('*').fillna(False).to_numpy().sum()))
print('전국 원본 공란 셀:', int(raw.isna().sum().sum()))
print('컬럼별 강남구 원본 결측치:')
display(missing_source.rename('missing_count').to_frame())
print('연령대 수치 기술통계:')
display(clean[AGE_COLS].describe().T)

강남구 정확히 동일한 행: 0
연도·시도·시군구 복합키 중복: 0
강남구 비식별(*) 셀: 0
강남구 원본 공란 셀: 0
잘못된/음수/소수 인원값이 있는 행: 0 0 0
검토 필요 행: 0
전국 원본 비식별(*) 셀: 113
전국 원본 공란 셀: 5
컬럼별 강남구 원본 결측치:


,missing_count
진료개시년도,0
시도,0
시도시군구,0
0-4세,0
5-9세,0
10-14세,0
15-19세,0
20-24세,0
25-29세,0
30-34세,0


연령대 수치 기술통계:


,count,mean,std,min,25%,50%,75%,max
0-4세,2.0,719.0,83.4386,660.0,689.5,719.0,748.5,778.0
5-9세,2.0,764.0,49.497475,729.0,746.5,764.0,781.5,799.0
10-14세,2.0,368.0,2.828427,366.0,367.0,368.0,369.0,370.0
15-19세,2.0,256.5,31.819805,234.0,245.25,256.5,267.75,279.0
20-24세,2.0,225.0,36.769553,199.0,212.0,225.0,238.0,251.0
25-29세,2.0,300.5,68.589358,252.0,276.25,300.5,324.75,349.0
30-34세,2.0,355.5,58.689863,314.0,334.75,355.5,376.25,397.0
35-39세,2.0,366.5,57.275649,326.0,346.25,366.5,386.75,407.0
40-44세,2.0,422.5,26.162951,404.0,413.25,422.5,431.75,441.0
45-49세,2.0,448.0,80.610173,391.0,419.5,448.0,476.5,505.0


## 3. 연령대 합산 및 연도별 변화

In [3]:
# 해당 연도·강남구의 18개 연령 구간을 그대로 합산하며, 고유 개인 수와 동일하다고 가정하지 않는다.
groups = {
    '0_19': ['0-4세', '5-9세', '10-14세', '15-19세'],
    '20_64': ['20-24세', '25-29세', '30-34세', '35-39세', '40-44세',
              '45-49세', '50-54세', '55-59세', '60-64세'],
    '65_plus': ['65-69세', '70-74세', '75-79세', '80-84세', '85세이상'],
}
for key, cols in groups.items():
    if not set(cols).issubset(AGE_COLS):
        raise ValueError(f'{key} 집계에 필요한 원본 컬럼 누락: {set(cols)-set(AGE_COLS)}')
    clean[f'age_count_{key}'] = clean[cols].sum(axis=1, min_count=len(cols))

clean['total_age_band_count'] = clean[AGE_COLS].sum(axis=1, min_count=len(AGE_COLS))
clean['observed_age_band_count'] = clean[AGE_COLS].notna().sum(axis=1)
for key in groups:
    clean[f'share_{key}'] = (
        clean[f'age_count_{key}']/clean['total_age_band_count'].replace(0, np.nan)
    ).astype('Float64')

# 연도간 비교는 원본에 실제 존재하는 연도에서만 계산하며, 연속된 연도만 전년 대비로 표시한다.
clean = clean.sort_values(['진료개시년도', 'source_row_id'], kind='stable').reset_index(drop=True)
clean['previous_year_total'] = clean['total_age_band_count'].shift(1)
clean['previous_year'] = clean['진료개시년도'].shift(1)
consecutive = clean['진료개시년도'].sub(clean['previous_year']).eq(1).fillna(False)
clean['year_over_year_abs'] = (clean['total_age_band_count']-clean['previous_year_total']).where(consecutive)
clean['year_over_year_pct'] = (
    100*clean['year_over_year_abs']/clean['previous_year_total'].replace(0, np.nan)
).astype('Float64')

shown = ['진료개시년도','데이터기준일','total_age_band_count',
         'age_count_0_19','age_count_20_64','age_count_65_plus',
         'suppressed_age_band_count','year_over_year_abs','year_over_year_pct']
display(clean[shown])
print('공간 좌표/행정동/250m 격자/도보 링크 컬럼: 없음 — 공간 해상도는 강남구 단위')
print('시간 단위: 연도만 존재(시각·일별 관측 없음)')

,진료개시년도,데이터기준일,total_age_band_count,age_count_0_19,age_count_20_64,age_count_65_plus,suppressed_age_band_count,year_over_year_abs,year_over_year_pct
0,2020,2025-01-01,7239.0,2059.0,3655.0,1525.0,0,<NA>,<NA>
1,2021,2025-01-01,6446.0,2156.0,2966.0,1324.0,0,-793.0,-10.954552


공간 좌표/행정동/250m 격자/도보 링크 컬럼: 없음 — 공간 해상도는 강남구 단위
시간 단위: 연도만 존재(시각·일별 관측 없음)


## 4. 시각화

In [ ]:
plot = clean.sort_values('진료개시년도')
years = plot['진료개시년도'].astype(str).tolist()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(years, plot['total_age_band_count'].astype(float))
ax.set(title='강남구 연령대별 천식 진료인원 합계', xlabel='연도', ylabel='연령대별 진료인원 합계')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show(); plt.close(fig)

x = np.arange(len(AGE_COLS))
width = 0.38
fig, ax = plt.subplots(figsize=(14, 5))
for i, (_, row) in enumerate(plot.iterrows()):
    vals = pd.to_numeric(row[AGE_COLS], errors='coerce').astype(float).to_numpy()
    ax.bar(x+(i-(len(plot)-1)/2)*width, vals, width=width, label=str(int(row['진료개시년도'])))
ax.set_xticks(x)
ax.set_xticklabels([c.replace('세이상', '세 이상') for c in AGE_COLS], rotation=60)
ax.set(title='강남구 연도·연령대별 천식 진료인원', xlabel='연령대', ylabel='진료인원')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show(); plt.close(fig)

# 히스토그램은 강남구의 연령 구간 값 18개씩의 분포를 비교하며, 개인별 분포는 아니다.
fig, ax = plt.subplots(figsize=(8, 4))
for _, row in plot.iterrows():
    vals = pd.to_numeric(row[AGE_COLS], errors='coerce').dropna().astype(float)
    ax.hist(vals, bins=8, alpha=0.5, label=str(int(row['진료개시년도'])))
ax.set(title='연령대별 진료인원 분포', xlabel='연령대별 진료인원', ylabel='연령대 수')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show(); plt.close(fig)

# 연령 구간별 전년 변화: 같은 연령 구간들을 짝지어 비교.
if len(plot)==2 and plot['진료개시년도'].iloc[1]-plot['진료개시년도'].iloc[0]==1:
    xval = pd.to_numeric(plot.iloc[0][AGE_COLS], errors='coerce').astype(float).to_numpy()
    yval = pd.to_numeric(plot.iloc[1][AGE_COLS], errors='coerce').astype(float).to_numpy()
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.scatter(xval, yval)
    bounds = [min(np.nanmin(xval),np.nanmin(yval)), max(np.nanmax(xval),np.nanmax(yval))]
    ax.plot(bounds, bounds, '--', color='gray', label='변화 없음')
    ax.set(xlabel=f'{years[0]}년 연령대별 진료인원', ylabel=f'{years[1]}년 연령대별 진료인원',
           title='연도별 연령대 진료인원 비교')
    ax.legend(); ax.grid(alpha=0.25)
    plt.tight_layout(); plt.show(); plt.close(fig)

## 5. summary.csv 및 features.csv 생성

In [5]:
rows = []
def summary_add(section, metric, category, value, unit='', note=''):
    rows.append({'section':section,'metric':metric,'category':category,'value':value,'unit':unit,'note':note})

for k,v in {
    'source_row_count':len(raw), 'source_column_count':len(raw_cols),
    'gangnam_row_count':len(clean), 'gangnam_year_count':clean['진료개시년도'].nunique(),
    'age_band_column_count':len(AGE_COLS),
    'gangnam_duplicate_record_rows':int(clean['duplicate_record_flag'].sum()),
    'gangnam_duplicate_key_rows':int(clean['duplicate_key_flag'].sum()),
    'gangnam_suppressed_cells':int(clean['suppressed_age_band_count'].sum()),
    'gangnam_review_needed_rows':int(clean['review_needed'].sum())
}.items():
    summary_add('overall', k, '', v, 'count')

for col in raw_cols:
    summary_add('missing_original', 'missing_count',col,int(selected[col].isna().sum()),'rows')
    summary_add('missing_original', 'missing_pct',col,float(selected[col].isna().mean()*100),'%')
for col in AGE_COLS:
    summary_add('suppression', 'suppressed_count', col, int(suppressed[col].sum()), 'cells')

for _, row in clean.iterrows():
    year=str(int(row['진료개시년도']))
    for col in AGE_COLS:
        summary_add('age_band_by_year','care_count',f'{year}|{col}',row[col],'persons_reported_by_age_band')
    for col in ['total_age_band_count','age_count_0_19','age_count_20_64','age_count_65_plus',
                'share_0_19','share_20_64','share_65_plus','year_over_year_abs','year_over_year_pct']:
        val=row[col]
        summary_add('year',col,year,val,
                    '%' if col=='year_over_year_pct' else ('ratio' if col.startswith('share_') else 'age_band_count'),
                    'Age-band sum; not independently verified as distinct persons' if 'count' in col else '')
for col in AGE_COLS:
    series=clean[col].astype('Float64')
    for stat in ['min','max','mean','median','std']:
        summary_add('age_band_statistics',stat,col,getattr(series,stat)(),'care_count')
summary=pd.DataFrame(rows)

# 강남구 1개 자치구, 두 개 연도 관측: 위치별 위험도 모델 입력으로 직접 확장하지 않는다.
features=pd.DataFrame({
    'source_row_id':clean['source_row_id'],
    'year':clean['진료개시년도'],
    'province':clean['시도'],
    'district':clean['시도시군구'],
    'data_reference_date':clean['데이터기준일'],
})
for col in AGE_COLS:
    ascii_name=col.replace('세이상','_plus').replace('세','').replace('-','_')
    features['age_'+ascii_name]=clean[col]
for col in ['observed_age_band_count','suppressed_age_band_count',
            'total_age_band_count','age_count_0_19','age_count_20_64','age_count_65_plus',
            'share_0_19','share_20_64','share_65_plus',
            'year_over_year_abs','year_over_year_pct','review_needed']:
    features[col]=clean[col]

print('clean.csv:',clean.shape)
print('summary.csv:',summary.shape)
print('features.csv:',features.shape)
display(features.head())

clean.csv: (2, 63)
summary.csv: (215, 6)
features.csv: (2, 35)


,source_row_id,year,province,district,data_reference_date,age_0_4,age_5_9,age_10_14,age_15_19,age_20_24,...,total_age_band_count,age_count_0_19,age_count_20_64,age_count_65_plus,share_0_19,share_20_64,share_65_plus,year_over_year_abs,year_over_year_pct,review_needed
0,23,2020,서울특별시,강남구,2025-01-01,660.0,799.0,366.0,234.0,251.0,...,7239.0,2059.0,3655.0,1525.0,0.284432,0.504904,0.210664,<NA>,<NA>,False
1,273,2021,서울특별시,강남구,2025-01-01,778.0,729.0,370.0,279.0,199.0,...,6446.0,2156.0,2966.0,1324.0,0.334471,0.46013,0.205399,-793.0,-10.954552,False


## 6. 결과 저장·재읽기 검증 및 README.md 생성

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
outputs={'clean.csv':clean,'summary.csv':summary,'features.csv':features}
loaded={}
for filename, frame in outputs.items():
    p=OUT_DIR/filename
    frame.to_csv(p,index=False,encoding='utf-8-sig')
    assert p.read_bytes().startswith(b'\xef\xbb\xbf'),f'BOM mismatch: {filename}'
    read_back=pd.read_csv(p,encoding='utf-8-sig',dtype='string')
    assert read_back.shape==frame.shape,(filename,'shape')
    assert list(read_back.columns)==frame.columns.tolist(),(filename,'columns')
    assert not any(str(x).startswith('Unnamed:') for x in read_back.columns)
    loaded[filename]=read_back

# 식별자와 핵심 연도 값 보존, 수치형 데이터 정확성 확인
for filename in ('clean.csv','features.csv'):
    saved=loaded[filename]
    original=outputs[filename]
    assert saved['source_row_id'].astype('int64').tolist()==original['source_row_id'].astype('int64').tolist()
for col in AGE_COLS:
    reread=pd.to_numeric(loaded['clean.csv'][col],errors='coerce')
    assert np.allclose(reread.to_numpy(dtype=float),clean[col].to_numpy(dtype=float),rtol=0,atol=0,equal_nan=True),col
for col in ('진료개시년도','시도','시도시군구','데이터기준일'):
    assert loaded['clean.csv'][col].fillna('').astype(str).tolist()==clean[col].fillna('').astype(str).tolist(),col
assert clean['시도'].eq('서울특별시').all() and clean['시도시군구'].eq('강남구').all()

# README는 실제 저장된 CSV 구조에서 자동 생성한다.
raw_descriptions={
    'source_row_id':'원본 전국 CSV 데이터 행 순서의 1-based 식별자(헤더 제외)',
    '진료개시년도':'천식 관련 진료가 개시된 연도',
    '시도':'광역 행정구역(서울특별시)',
    '시도시군구':'원본 시군구(강남구)',
    '데이터기준일':'원본 데이터 기준일(진료개시년도와는 다른 날짜)',
    'valid_year_flag':'진료개시년도 형식 및 기준일 이후 연도가 아닌지 검증',
    'valid_reference_date_flag':'데이터기준일 날짜 파싱 여부',
    'duplicate_record_flag':'원본 전체 컬럼 기준 중복 행 플래그(중복 제거하지 않음)',
    'duplicate_key_flag':'진료개시년도+시도+시도시군구 복합키 반복 플래그',
    'suppressed_age_band_count':'연령대 원본 비식별(*) 항목 개수',
    'missing_age_band_count':'비식별·공란·비수치 값을 포함한 연령대 결측 개수',
    'invalid_age_count_flag':'비식별(*) 및 정상 공란 이외의 숫자 변환 불가 항목 존재 여부',
    'negative_age_count_flag':'음수 연령대별 진료인원 존재 여부',
    'non_integer_age_count_flag':'정수가 아닌 연령대별 진료인원 존재 여부',
    'review_needed':'연도·날짜·중복키·잘못된 숫자값 관련 검토 대상 여부',
    'total_age_band_count':'모든 연령대 인원값이 있을 때만 계산한 18개 연령구간 합(고유 환자수와 동일하다고 단정 불가)',
    'observed_age_band_count':'수치형으로 관측된 연령 구간 수',
    'age_count_0_19':'0~19세 원본 연령구간의 합(해당 그룹 전부 관측된 경우)',
    'age_count_20_64':'20~64세 원본 연령구간의 합(해당 그룹 전부 관측된 경우)',
    'age_count_65_plus':'65세 이상 원본 연령구간의 합(해당 그룹 전부 관측된 경우)',
    'share_0_19':'0~19세 합 / 18개 연령 구간 합',
    'share_20_64':'20~64세 합 / 18개 연령 구간 합',
    'share_65_plus':'65세 이상 합 / 18개 연령 구간 합',
    'previous_year_total':'전년의 연령 구간 합(연속 연도 비교 시만 활용)',
    'previous_year':'직전 기록 연도',
    'year_over_year_abs':'연속된 전년 대비 연령 구간 합의 증감',
    'year_over_year_pct':'연속된 전년 대비 연령 구간 합의 증감률(%)',
    'section':'EDA 집계 영역',
    'metric':'집계 지표',
    'category':'집계 범주 또는 원본 컬럼명',
    'value':'실제 계산된 값',
    'unit':'측정 단위 또는 비율',
    'note':'집계값 해석 관련 설명',
    'year':'원본 진료개시년도',
    'province':'원본 시도',
    'district':'원본 시도시군구',
    'data_reference_date':'원본 데이터기준일',
}

def column_description(col):
    if col in raw_descriptions: return raw_descriptions[col]
    if col in AGE_COLS: return f'원본 연령구간 {col}의 천식 관련 진료인원; * 비식별값은 결측으로 보존'
    if col.endswith('_suppressed_flag'):
        return f'{col.removesuffix("_suppressed_flag")}의 원본 * 비식별 여부'
    if col.startswith('age_'):
        return f'원본 연령대별 진료인원을 ASCII 컬럼명으로 전환: {col}'
    return '원본 컬럼 기반 값; 추가 의미 검증 필요'

def escape_md(value):
    return str(value).replace('|','\\|').replace('\n',' ')

lines=[
    '# 강남구 천식 관련 진료인원 EDA 산출물', '',
    f'- 데이터셋: `{DATASET_KEY}`',
    f'- 원본 파일: `{RAW_PATH.name}`',
    f'- 원본 인코딩: `{encoding}`',
    f'- 전국 원본: {len(raw):,}행 × {len(raw_cols)}열',
    f'- 강남구 원본 추출: {len(selected):,}행 × {len(raw_cols)}열',
    '- 필터: `시도 == 서울특별시` AND `시도시군구 == 강남구`',
    '- 분석 단위: 연도 × 강남구(좌표/격자/도보 링크 없음)',
    '- 원본 수정·삭제 없음, 다른 자료와 결합 없음',
    '', '## 처리 및 주의사항', '',
    '- 연령대별 수치 컬럼을 수치형으로 변환하고 원본 `*`는 결측 처리하되 비식별 플래그로 보존한다.',
    '- 0을 비식별값 대체치로 사용하지 않으며, 불확실한 중복 행도 임의로 제거하지 않는다.',
    '- 그룹별 합계와 전체 합계는 포함된 연령구간이 모두 관측된 경우에만 계산한다.',
    '- 진료인원 연령구간 합은 원본 18개 인원값을 합산한 결과로 고유 환자 수 또는 천식 유병률을 의미한다고 단정할 수 없다.',
    '- 데이터는 천식 관련 **진료인원**이며 실제 흡연 발생, 간접흡연 노출량, 모델 정답 y가 아니다.',
    '- 2020·2021년 강남구 연도별 2행으로, 2026년 실시간·도보 링크별 모델 X로 직접 일반화할 수 없다.',
    '- 인구 분모·표준화 연령 구조가 없어 지역 간 환자 발생률·표준화 유병률 계산은 불가능하다.',
    '- 시간대·날짜별 관측값·위경도·250m 격자가 없어 이를 임의 생성하지 않았다.',
    '', '## 파일 간 연결', '',
    '- `clean.csv`와 `features.csv`: `source_row_id`로 1:1 연결',
    '- 행정·시간별 비교 키: `진료개시년도` + `시도` + `시도시군구` (features의 경우 `year` + `province` + `district`)',
    '- `summary.csv`는 여러 집계 수준의 테이블이므로 개별 행과 1:1 조인하지 않는다.',
    '', '## CSV별 실제 컬럼 명세', ''
]
file_desc={'clean.csv':'원본 정보 보존·자료형 및 품질 검증·파생값 포함',
           'summary.csv':'원본 결측·비식별·연도·연령대별 실제 EDA 집계',
           'features.csv':'강남구 연도 단위의 건강 취약성 참고 변수 후보'}
for filename, frame in outputs.items():
    lines.extend([f'### {filename}', '', f'- 목적: {file_desc[filename]}',
                  f'- 구조: {len(frame):,}행 × {len(frame.columns)}열',
                  '- 인코딩: UTF-8 with BOM (`utf-8-sig`), 인덱스 미저장',
                  '', '| 컬럼명 | 저장 후 자료형 | 의미·생성 방식 |', '|---|---|---|'])
    reread=loaded[filename]
    for col in reread.columns:
        # 저장된 CSV는 자료형 메타데이터가 없으므로 저장 전 pandas dtype도 병기
        lines.append(f'| `{escape_md(col)}` | `{frame[col].dtype}` (원본 pandas) | {escape_md(column_description(col))} |')
    lines.append('')
lines.extend([
    '## 재실행 및 검증', '',
    '- `do_not_lung/notebooks/01_eda/EDA_국민건강보험공단_연도별 시군구별 천식 관련 진료인원_20250101.ipynb` 실행',
    '- 원본은 `data/raw_data` 아래에 두고 저장소 또는 상위폴더에서 Notebook을 실행한다.',
    '- CSV는 `data/processed_data/국민건강보험공단_연도별 시군구별 천식 관련 진료인원_20250101`에 자동 저장한다.',
    '- 각 CSV 재읽기 시 BOM, 행·열 수, 컬럼, 원본 식별자·연령대별 수치를 검증한다.',
    '- README의 컬럼 명세는 실제 저장한 CSV의 컬럼명과 전처리 후 pandas 자료형을 통해 자동 생성한다.',
])
readme_path=OUT_DIR/'README.md'
readme_path.write_text('\n'.join(lines)+'\n',encoding='utf-8')
readback=readme_path.read_text(encoding='utf-8')
for frame in outputs.values():
    for col in frame.columns:
        assert f'`{col}`' in readback, col
print('CSV/README 생성 및 검증 완료')
for p in sorted(OUT_DIR.iterdir()):
    print(p.name, p.stat().st_size, 'bytes')

CSV/README 생성 및 검증 완료
clean.csv 1960 bytes
features.csv 982 bytes
README.md 15235 bytes
summary.csv 12614 bytes
